# CivicFlow — Multi-Step GRPO Training

**What the old approach did wrong:** The previous reward function reset the environment, stepped it *once*, and returned the step reward. This trained the model to pick a good *first* action from a blank slate — not to plan a full episode.

**What this notebook does instead:**

For each GRPO completion (the model's proposed first action), the reward function:
1. Applies that action to the environment
2. Runs the **remaining steps of the episode** using the same model (inference only, no gradient)
3. Returns the **total cumulative reward** across the full episode

The GRPO gradient still only flows through the **first action** (that's what GRPOTrainer generates and scores). But the reward signal now measures full-episode quality, so the model is pushed toward actions that lead to good long-term outcomes — not just good single steps.

```
GRPO generates: action_0  (gradient here)
Reward fn runs: action_1, action_2, ... action_N  (inference only, no gradient)
Reward signal:  sum(r_0 + r_1 + ... + r_N)  <- full episode quality
```

**Prerequisites:**
- Run the SFT Colab first → push `Aaryan369/civicflow-sft-qwen2.5-3b` to Hub
- Runtime → Change runtime type → **T4 GPU**
- Expected training time: ~40 min for 60 steps on T4

In [ ]:
# Cell 1 — Install
!pip install -q unsloth trl datasets pydantic openenv-core
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
print('Done')

In [ ]:
# Cell 2 — Clone CivicFlow environment
import subprocess
result = subprocess.run(
    ['git', 'clone', 'https://github.com/Aaryan-Antala/Meta_x_HF.git', 'civicflow_repo'],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)

import sys
sys.path.insert(0, 'civicflow_repo')

from civicflow_env.models import CivicflowAction
from civicflow_env.server.civicflow_env_environment import CivicflowEnvironment
from civicflow_env.tasks import list_task_ids
print('Task IDs:', list_task_ids())

In [ ]:
# Cell 3 — Load SFT-warmed model with Unsloth
import torch
from unsloth import FastLanguageModel

SFT_MODEL   = 'Aaryan369/civicflow-sft-qwen2.5-3b'
MAX_SEQ_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=SFT_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print('Model loaded. GPU memory:', torch.cuda.memory_allocated() / 1e9, 'GB')

In [ ]:
# Cell 4 — Multi-step reward function
#
# The reward function runs a full episode:
#   step 0  : action from GRPO's completion  (this is what GRPO optimises)
#   steps 1+ : actions from model inference  (no gradient, but shapes the reward)
#
# Using model inference for steps 1+ means the reward reflects how good
# the first action is AS A STARTING POINT for the model's own continuation.
# This is the key difference from the naive single-step approach.

import json, os, re

SYSTEM_PROMPT = (
    'You are a municipal planner. Given the current city state, emit exactly one '
    'planning action as a JSON object. Use only keys from legal_actions_summary. '
    'Respond with only the JSON object — no explanation, no markdown fences.'
)

MAX_ROLLOUT_STEPS = 10   # cap per episode (tiny tasks need ~14; we cap for speed)


def _extract_json(text: str) -> str:
    """Pull the first {...} object out of any model output format."""
    # Strip <think>...</think> reasoning blocks (Qwen3 style)
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL)
    text = re.sub(r'<think>.*$',         '', text, flags=re.DOTALL)
    # Strip markdown fences
    text = re.sub(r'```[a-z]*\n?', '', text)
    # Find first {...}
    m = re.search(r'\{[^{}]*\}', text, re.DOTALL)
    return m.group(0).strip() if m else text.strip()


def _build_user_payload(obs) -> str:
    payload = {
        'current_phase':        obs.current_phase,
        'phase_objective':      obs.phase_objective,
        'observation_summary':  obs.planning_summary,
        'active_constraints':   obs.active_constraints,
        'legal_actions_summary': obs.legal_actions_summary,
    }
    return json.dumps(payload, separators=(',', ':'))


@torch.no_grad()
def _model_generate_action(obs) -> str:
    """Inference-only generation used for rollout steps 1+.
    torch.no_grad() ensures no gradients accumulate here.
    """
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': _build_user_payload(obs)},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=1800).to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=False,                           # greedy — fast and deterministic
        pad_token_id=tokenizer.eos_token_id,
    )
    new_tokens = out[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


def civicflow_reward_fn(completions, task_ids, **kwargs):
    """Multi-step reward: runs the full episode, not just one step."""
    rewards = []

    for completion, task_id in zip(completions, task_ids):
        # GRPO passes completions as list-of-messages or raw string
        first_raw = completion[0]['content'] if isinstance(completion, list) else completion

        # ── Set up fresh episode ──────────────────────────────────────────────
        os.environ['CIVICFLOW_TASK_ID'] = task_id
        env = CivicflowEnvironment()
        obs = env.reset()
        total_reward = 0.0

        # ── Step 0: GRPO's completion (gradient flows through this action) ───
        try:
            action = CivicflowAction(**json.loads(_extract_json(first_raw)))
            obs = env.step(action)
            total_reward += obs.reward
        except Exception:
            rewards.append(-1.0)   # format penalty
            continue

        # ── Steps 1+: model inference (no gradient, shapes the reward signal) ─
        for _ in range(MAX_ROLLOUT_STEPS - 1):
            if env._done:
                break
            raw = _model_generate_action(obs)
            try:
                action = CivicflowAction(**json.loads(_extract_json(raw)))
                obs = env.step(action)
                total_reward += obs.reward
            except Exception:
                total_reward -= 0.5   # parse error mid-episode
                break

        # Bonus for completing the plan within the rollout window
        if obs.last_metrics.get('final_valid_plan'):
            total_reward += 5.0

        rewards.append(total_reward)

    return rewards


# ── Sanity check: reward function runs without crash ─────────────────────────
test_completion = json.dumps({'action_type': 'set_zoning', 'block_id': 'B1', 'zone': 'residential'})
r = civicflow_reward_fn([[{'role':'assistant','content': test_completion}]], ['tiny_a'])
print(f'Sanity reward: {r[0]:.3f}  (non-crash + non-zero = good)')

In [ ]:
# Cell 5 — Build prompt dataset
#
# Each row = the INITIAL observation of a tiny task.
# GRPO will generate candidate first-actions, then the reward function
# runs the full episode from there.
#
# We repeat tasks 30x each for batch diversity across training steps.

from datasets import Dataset

def make_prompt(task_id: str) -> dict:
    os.environ['CIVICFLOW_TASK_ID'] = task_id
    env = CivicflowEnvironment()
    obs = env.reset()
    return {
        'prompt': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': _build_user_payload(obs)},
        ],
        'task_id': task_id,
    }

TRAIN_TASKS = ['tiny_a', 'tiny_b', 'tiny_c']
rows = [make_prompt(t) for t in TRAIN_TASKS * 30]   # 90 rows
dataset = Dataset.from_list(rows)
print(f'Dataset: {len(dataset)} rows across {len(TRAIN_TASKS)} tasks')

In [ ]:
# Cell 6 — GRPO training
#
# Config notes:
#   num_generations=2     : 2 candidate actions per prompt (4 would OOM on T4 with episode rollouts)
#   max_new_tokens=64     : actions are short JSON, no need for 128+
#   per_device_train_batch_size=1 : episode rollouts are memory-heavy
#   gradient_accumulation_steps=4 : effective batch = 4, keeps training stable
#   learning_rate=3e-6    : small — we're fine-tuning an already-SFT model

from trl import GRPOTrainer, GRPOConfig

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[civicflow_reward_fn],
    args=GRPOConfig(
        output_dir='civicflow_grpo_checkpoints',
        num_train_epochs=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        learning_rate=3e-6,
        max_new_tokens=64,
        num_generations=2,
        logging_steps=1,
        save_steps=20,
        max_grad_norm=0.3,
        warmup_ratio=0.05,
        report_to='none',
        seed=42,
    ),
    train_dataset=dataset,
)

print('Starting GRPO training with multi-step episode rollouts...')
print(f'Each training step: model generates 2 actions → reward fn runs up to {MAX_ROLLOUT_STEPS} steps each')
trainer.train()

In [ ]:
# Cell 7 — Quick evaluation: run the trained model on tiny_a and print the episode
import textwrap

FastLanguageModel.for_inference(model)

def run_greedy_episode(task_id, max_steps=20):
    os.environ['CIVICFLOW_TASK_ID'] = task_id
    env = CivicflowEnvironment()
    obs = env.reset()
    total, illegal, steps = 0.0, 0, 0
    print(f'\n=== {task_id} ===')
    for step in range(max_steps):
        raw = _model_generate_action(obs)
        try:
            action = CivicflowAction(**json.loads(_extract_json(raw)))
        except Exception:
            print(f'  step {step:>2}: PARSE ERROR | {raw[:60]}')
            illegal += 1
            break
        obs = env.step(action)
        total += obs.reward
        flag = 'ILLEGAL' if obs.last_metrics.get('illegal_action') else 'ok'
        if obs.last_metrics.get('illegal_action'): illegal += 1
        print(f'  step {step:>2} [{flag:>7}] r={obs.reward:+.3f} prog={obs.last_metrics["progress_score"]:.2f} | '
              f'{action.action_type}({action.block_id or action.infra_zone or ""})')
        steps += 1
        if env._done: break
    print(f'  RESULT: valid={obs.last_metrics["final_valid_plan"]} '
          f'progress={obs.last_metrics["progress_score"]:.2f} '
          f'total_reward={total:+.3f} illegal={illegal}')
    return {'valid': obs.last_metrics['final_valid_plan'],
            'progress': obs.last_metrics['progress_score'],
            'reward': total, 'illegal': illegal, 'steps': steps}

results = [run_greedy_episode(t) for t in ['tiny_a', 'tiny_b', 'tiny_c']]

In [ ]:
# Cell 8 — Save and push
HF_USERNAME = 'Aaryan369'
GRPO_REPO   = f'{HF_USERNAME}/civicflow-grpo-qwen2.5-3b'

model.save_pretrained('civicflow_grpo_adapter')
tokenizer.save_pretrained('civicflow_grpo_adapter')

model.save_pretrained_merged('civicflow_grpo_merged', tokenizer, save_method='merged_16bit')

model.push_to_hub(GRPO_REPO)
tokenizer.push_to_hub(GRPO_REPO)
print(f'Pushed → https://huggingface.co/{GRPO_REPO}')

In [ ]:
# Cell 9 — Reward curve + episode summary plots
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

log = trainer.state.log_history

# Pull reward and loss from log history
reward_keys = ['reward', 'rewards/mean', 'train/reward']
loss_keys   = ['loss', 'train/loss', 'policy_loss']

reward_series = [(x['step'], x[k]) for x in log for k in reward_keys if k in x]
loss_series   = [(x['step'], x[k]) for x in log for k in loss_keys   if k in x]

fig = plt.figure(figsize=(14, 5))
gs  = gridspec.GridSpec(1, 3, figure=fig)

# Plot 1: reward curve
ax1 = fig.add_subplot(gs[0, :2])
if reward_series:
    steps, vals = zip(*reward_series)
    ax1.plot(steps, vals, color='green', linewidth=1.5, label='episode reward (multi-step)')
    # rolling average
    window = max(1, len(vals)//10)
    avg = [sum(vals[max(0,i-window):i+1])/len(vals[max(0,i-window):i+1]) for i in range(len(vals))]
    ax1.plot(steps, avg, color='darkgreen', linewidth=2.5, label=f'rolling avg ({window}-step)')
ax1.axhline(0, color='red', linestyle='--', alpha=0.4, label='zero baseline')
ax1.set_xlabel('Training step')
ax1.set_ylabel('Cumulative episode reward')
ax1.set_title('CivicFlow GRPO — multi-step episode reward')
ax1.legend()

# Plot 2: post-training episode results
ax2 = fig.add_subplot(gs[0, 2])
task_names = ['tiny_a', 'tiny_b', 'tiny_c']
progress   = [r['progress'] for r in results]
colors     = ['#2ecc71' if r['valid'] else '#e74c3c' for r in results]
bars = ax2.bar(task_names, progress, color=colors, edgecolor='black', linewidth=0.5)
ax2.set_ylim(0, 1.1)
ax2.axhline(1.0, color='green', linestyle='--', alpha=0.5, label='perfect (1.0)')
ax2.set_ylabel('Progress score')
ax2.set_title('Post-training progress\n(green = valid plan)')
for bar, r in zip(bars, results):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f"{r['progress']:.2f}", ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('grpo_reward_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved grpo_reward_curve.png')

# Summary table
print('\nPost-training evaluation:')
print(f'{"task":<10} {"valid":>5} {"progress":>9} {"reward":>9} {"illegal":>8}')
for t, r in zip(task_names, results):
    print(f'{t:<10} {r["valid"]:>5} {r["progress"]:>9.2f} {r["reward"]:>+9.3f} {r["illegal"]:>8}')

In [ ]:
# Cell 10 — Download plot
# Commit this to civicflow_env/assets/grpo_reward_curve.png
from google.colab import files
files.download('grpo_reward_curve.png')
print('Download complete — commit to civicflow_env/assets/grpo_reward_curve.png')